In [28]:
import requests
import time
import pandas as pd

In [29]:
ANO_INICIO_BASE = 2025
ANO_ATUAL = 2025

In [30]:
def quali():
    lista_qualis = []
    for i in range(ANO_INICIO_BASE, (ANO_ATUAL + 1)):
        rodada = 1
        while True:
            acesso = requests.get(f"https://api.jolpi.ca/ergast/f1/{i}/{rodada}/qualifying/").json()
            races = acesso["MRData"]["RaceTable"]["Races"]

            if not races:
                break
            else:
                for race in races:
                    qualificatorias = race["QualifyingResults"]
                    circuito = race["Circuit"]["circuitId"]

                    for qualificatoria in qualificatorias:
                        lista_qualis.append({
                            "temporada_atual": acesso["MRData"]["RaceTable"]["season"],
                            "rodada_atual": acesso["MRData"]["RaceTable"]["round"],
                            "id_circuito_atual": circuito,
                            "id_piloto_atual": qualificatoria["Driver"]["driverId"],
                            "id_equipe_atual": qualificatoria["Constructor"]["constructorId"],
                            "posicao_quali_atual": qualificatoria.get("position", None),
                            "q1_atual": qualificatoria.get("Q1", None),
                            "q2_atual": qualificatoria.get("Q2", None),
                            "q3_atual": qualificatoria.get("Q3", None)
                        })
                rodada += 1
            time.sleep(1)
                    
    
    return lista_qualis


qualis = quali()
df = pd.DataFrame(qualis)
df_quali = df.astype({"temporada_atual": int, "rodada_atual": int}).sort_values(by=["temporada_atual", "rodada_atual"], ascending=True)

In [31]:
def resultados():
    lista_resultados = []
    for i in range(ANO_INICIO_BASE, (ANO_ATUAL + 1)):
        rodada = 1
        while True:
            acesso = requests.get(f"https://api.jolpi.ca/ergast/f1/{i}/{rodada}/results/").json()
            races = acesso["MRData"]["RaceTable"]["Races"]

            if not races:
                break
            else:
                for race in races:
                    results = race["Results"]
                    circuito = race["Circuit"]["circuitId"]

                    for result in results:
                        lista_resultados.append({
                            "temporada_atual": acesso["MRData"]["RaceTable"]["season"],
                            "rodada_atual": acesso["MRData"]["RaceTable"]["round"],
                            "id_circuito_atual": circuito,
                            "id_piloto_atual": result["Driver"]["driverId"],
                            "id_equipe_atual": result["Constructor"]["constructorId"],
                            "posicao_corrida_anterior": result["positionText"],
                            "posicao_ultima_corrida": result["position"],
                            "target": result["position"],
                            "grid_anterior": result["grid"],
                            "status": result["status"],
                            "pontos_anterior_individual": result["points"]
                        })
                rodada += 1
            time.sleep(1)
                    
    
    return lista_resultados


results = resultados()
df_results = pd.DataFrame(results)
df_results = df_results.astype({"temporada_atual": int, "rodada_atual": int}).sort_values(by=["temporada_atual", "rodada_atual"], ascending=True)

In [32]:
def resultados_equipes():
    lista_resultados = []
    for i in range(ANO_INICIO_BASE, (ANO_ATUAL + 1)):
        rodada = 1
        while True:
            acesso = requests.get(f"https://api.jolpi.ca/ergast/f1/{i}/{rodada}/constructorstandings/").json()
            standingsLists = acesso["MRData"]["StandingsTable"]["StandingsLists"]

            if not standingsLists:
                break
            else:
                for standings in standingsLists:
                    construtors = standings["ConstructorStandings"]
                    for constructor in construtors:
                        lista_resultados.append({
                            "temporada_atual": standings["season"],
                            "rodada_atual": standings["round"],
                            "posicao_equipe_anterior": constructor["positionText"],
                            "pontos_equipe_anterior": constructor["points"],
                            "vitorias_equipe_anterior": constructor["wins"],
                            "id_equipe_atual": constructor["Constructor"]["constructorId"]
                            })
                rodada += 1
            time.sleep(1)
                    
    
    return lista_resultados


results = resultados_equipes()
df_teams = pd.DataFrame(results)
df_teams = df_teams.astype({"temporada_atual": int, "rodada_atual": int}).sort_values(by=["temporada_atual", "rodada_atual"], ascending=True)

In [33]:
df = pd.merge(
    df_quali,
    df_results,
    on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
    how="outer"
)
df["id_circuito_atual"] = df["id_circuito_atual_x"].combine_first(df["id_circuito_atual_y"])
df["id_equipe_atual"] = df["id_equipe_atual_x"].combine_first(df["id_equipe_atual_y"])
df.drop(columns=["id_circuito_atual_x", "id_circuito_atual_y", "id_equipe_atual_x", "id_equipe_atual_y"], inplace=True)
df.reset_index(drop=True,inplace=True)

In [34]:
df_pos_anterior = df_results[["temporada_atual", "rodada_atual", "id_piloto_atual", "posicao_corrida_anterior", "grid_anterior", "posicao_ultima_corrida", "status", "pontos_anterior_individual"]].copy()
df_pos_anterior["rodada_atual"] = df_pos_anterior["rodada_atual"] + 1

df = pd.merge(
    df,
    df_pos_anterior.rename(columns={"posicao_corrida_anterior": "posicao_anterior"}),
    on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
    how="left"
)

In [35]:
df_pos_anterior_equipe = df_teams[["temporada_atual", "rodada_atual", "id_equipe_atual", "posicao_equipe_anterior", "pontos_equipe_anterior", "vitorias_equipe_anterior"]].copy()
df_pos_anterior_equipe["rodada_atual"] = df_pos_anterior_equipe["rodada_atual"] + 1

df = pd.merge(
    df,
    df_pos_anterior_equipe,
    on=["temporada_atual", "rodada_atual", "id_equipe_atual"],
    how="left"
)

In [36]:
print(df.columns.tolist())

['temporada_atual', 'rodada_atual', 'id_piloto_atual', 'posicao_quali_atual', 'q1_atual', 'q2_atual', 'q3_atual', 'posicao_corrida_anterior', 'posicao_ultima_corrida_x', 'target', 'grid_anterior_x', 'status_x', 'pontos_anterior_individual_x', 'id_circuito_atual', 'id_equipe_atual', 'posicao_anterior', 'grid_anterior_y', 'posicao_ultima_corrida_y', 'status_y', 'pontos_anterior_individual_y', 'posicao_equipe_anterior', 'pontos_equipe_anterior', 'vitorias_equipe_anterior']


In [37]:
temporadas = df["temporada_atual"].unique()

In [38]:
ultimo_round = df['rodada_atual']
ultimo_round = ultimo_round.max()


In [39]:
def tempo_para_ms(tempo_str):
    if pd.isna(tempo_str):
        return None
    try:
        minutos, resto = tempo_str.split(':')
        segundos, ms = resto.split('.')
        return int(minutos) * 60000 + int(segundos) * 1000 + int(ms)
    except:
        return None

df['q1_atual'] = df['q1_atual'].apply(tempo_para_ms)
df['q2_atual'] = df['q2_atual'].apply(tempo_para_ms)
df['q3_atual'] = df['q3_atual'].apply(tempo_para_ms)

In [40]:
q3_rodada = []
for temporada in temporadas:
    for rodada in range(1, (ultimo_round + 1)):
        df_delta = df[(df['temporada_atual'] == temporada) & (df['rodada_atual'] == rodada)]
        if len(df_delta) > 0:
            menor_tempo = df_delta["q3_atual"].min()
            for value_q3 in df_delta['q3_atual']:
                if pd.notna(value_q3) and value_q3 != '':
                    dif = value_q3 - menor_tempo
                    q3_rodada.append(dif)
                else:
                    q3_rodada.append(None)

df["dif_para_pole_atual"] = q3_rodada

In [41]:
lista_pontuacao = []

for data in range(ANO_INICIO_BASE, (ANO_ATUAL + 1)):
    rodada = 1
    while True:
        acesso = requests.get(f"https://api.jolpi.ca/ergast/f1/{data}/{rodada}/driverstandings/").json()
        StandingsLists = acesso["MRData"]["StandingsTable"]["StandingsLists"]
        
        if not StandingsLists:
            break
        else:
            for DriverStanding in StandingsLists:
                for pontos in DriverStanding["DriverStandings"]:
                    df_posicao = df_results[(df_results["rodada_atual"] <= rodada) & (df_results["temporada_atual"] == data) & (df_results["id_piloto_atual"] == pontos["Driver"]["driverId"])]["posicao_ultima_corrida"]
                    grid = df_results[(df_results["rodada_atual"] <= rodada) & (df_results["temporada_atual"] == data) & (df_results["id_piloto_atual"] == pontos["Driver"]["driverId"])]["grid_anterior"]
                    media_3 = pd.to_numeric(df_posicao, errors="coerce").tail(3).mean()
                    media_5 = pd.to_numeric(df_posicao, errors="coerce").tail(5).mean()

                    lista_pontuacao.append({
                        "pontos_anterior": pontos["points"],
                        "rodada_atual": DriverStanding["round"],
                        "temporada_atual": DriverStanding["season"],
                        "id_piloto_atual": pontos["Driver"]["driverId"],
                        "posicao_camp_anterior": pontos["positionText"],
                        "num_vitorias_anterior": pontos["wins"],
                        "media_ultimas_3_anterior": f"{media_3:.2f}",
                        "media_ultimas_5_anterior": f"{media_5:.2f}",
                        "qtde_abandonos_anterior": f"{df[(df["temporada_atual"] == data) & (df["rodada_atual"] <= rodada) & (df["id_piloto_atual"] == pontos["Driver"]["driverId"]) & (df["posicao_anterior"] == 'R')].shape[0]}",
                        "media_posicao_ganha_anterior": f"{( pd.to_numeric(grid, errors="coerce").tail(3) - pd.to_numeric(df_posicao, errors="coerce").tail(3) ).mean():.2f}",
                        "tendencia_desempenho": f"{media_3 - media_5:.2f}"
                    })
            rodada += 1
        time.sleep(1)

df_pontos = pd.DataFrame(lista_pontuacao)
df_pontos = df_pontos.astype({"temporada_atual": int, "rodada_atual": int}).sort_values(by=["temporada_atual", "rodada_atual"], ascending=True)

In [42]:
df["rodada_anterior"] = df["rodada_atual"] - 1

df = pd.merge(
    df,
    df_pontos,
    left_on=["temporada_atual", "rodada_anterior", "id_piloto_atual"],
    right_on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
    how="left"
)

In [43]:
cols_medias = [
    'media_ultimas_3_anterior',
    'media_ultimas_5_anterior', 
    'media_posicao_ganha_anterior',
    'tendencia_desempenho'
]

for col in cols_medias:
    df[col] = df[col].replace('nan', pd.NA)
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df[cols_medias].isnull().sum())
print(df[cols_medias].dtypes)

media_ultimas_3_anterior        21
media_ultimas_5_anterior        21
media_posicao_ganha_anterior    21
tendencia_desempenho            21
dtype: int64
media_ultimas_3_anterior        float64
media_ultimas_5_anterior        float64
media_posicao_ganha_anterior    float64
tendencia_desempenho            float64
dtype: object


In [44]:
print(df.columns.tolist())

['temporada_atual', 'rodada_atual_x', 'id_piloto_atual', 'posicao_quali_atual', 'q1_atual', 'q2_atual', 'q3_atual', 'posicao_corrida_anterior', 'posicao_ultima_corrida_x', 'target', 'grid_anterior_x', 'status_x', 'pontos_anterior_individual_x', 'id_circuito_atual', 'id_equipe_atual', 'posicao_anterior', 'grid_anterior_y', 'posicao_ultima_corrida_y', 'status_y', 'pontos_anterior_individual_y', 'posicao_equipe_anterior', 'pontos_equipe_anterior', 'vitorias_equipe_anterior', 'dif_para_pole_atual', 'rodada_anterior', 'pontos_anterior', 'rodada_atual_y', 'posicao_camp_anterior', 'num_vitorias_anterior', 'media_ultimas_3_anterior', 'media_ultimas_5_anterior', 'qtde_abandonos_anterior', 'media_posicao_ganha_anterior', 'tendencia_desempenho']


In [45]:
df['posicao_camp_anterior'] = pd.to_numeric(df['posicao_camp_anterior'], errors='coerce')

In [46]:
print(df.columns.tolist())

['temporada_atual', 'rodada_atual_x', 'id_piloto_atual', 'posicao_quali_atual', 'q1_atual', 'q2_atual', 'q3_atual', 'posicao_corrida_anterior', 'posicao_ultima_corrida_x', 'target', 'grid_anterior_x', 'status_x', 'pontos_anterior_individual_x', 'id_circuito_atual', 'id_equipe_atual', 'posicao_anterior', 'grid_anterior_y', 'posicao_ultima_corrida_y', 'status_y', 'pontos_anterior_individual_y', 'posicao_equipe_anterior', 'pontos_equipe_anterior', 'vitorias_equipe_anterior', 'dif_para_pole_atual', 'rodada_anterior', 'pontos_anterior', 'rodada_atual_y', 'posicao_camp_anterior', 'num_vitorias_anterior', 'media_ultimas_3_anterior', 'media_ultimas_5_anterior', 'qtde_abandonos_anterior', 'media_posicao_ganha_anterior', 'tendencia_desempenho']


In [47]:
df.drop(
    columns=[
        "rodada_atual_y",
        "rodada_anterior",
        "grid_anterior_x",
    ],
    inplace=True,
)

df.rename(
    columns={
        "rodada_atual_x": "rodada_atual",
        "posicao_corrida_anterior": "posicao_corrida_atual",
        "grid_anterior_y": "grid_anterior",
    },
    inplace=True,
)

In [48]:
df = df.dropna(subset=['posicao_corrida_atual']).reset_index(drop=True)
print(df.shape)

(479, 31)


In [49]:
print(df.columns.tolist())

['temporada_atual', 'rodada_atual', 'id_piloto_atual', 'posicao_quali_atual', 'q1_atual', 'q2_atual', 'q3_atual', 'posicao_corrida_atual', 'posicao_ultima_corrida_x', 'target', 'status_x', 'pontos_anterior_individual_x', 'id_circuito_atual', 'id_equipe_atual', 'posicao_anterior', 'grid_anterior', 'posicao_ultima_corrida_y', 'status_y', 'pontos_anterior_individual_y', 'posicao_equipe_anterior', 'pontos_equipe_anterior', 'vitorias_equipe_anterior', 'dif_para_pole_atual', 'pontos_anterior', 'posicao_camp_anterior', 'num_vitorias_anterior', 'media_ultimas_3_anterior', 'media_ultimas_5_anterior', 'qtde_abandonos_anterior', 'media_posicao_ganha_anterior', 'tendencia_desempenho']


In [50]:
df_clima = pd.read_csv(r"DATA\clima_f1.csv", sep=";")
df_clima.rename(columns={"temporada": "temporada_atual", "rodada": "rodada_atual", "posicao_ultima_corrida_y": "posicao_ultima_corrida"}, inplace=True)
df_clima.head()

,temporada_atual,rodada_atual,temp_ar_media,temp_pista_media,umidade_media,corrida_molhada,perc_voltas_chuva
0,2018,1,24.08,36.32,30.92,0,4.5
1,2018,2,27.98,32.20,47.36,0,0.0
2,2018,3,19.45,37.02,24.09,0,1.8
3,2018,4,16.66,25.25,45.65,0,0.0
4,2018,5,16.05,32.34,52.29,1,79.0


In [51]:
df = pd.merge(
    df,
    df_clima,
    on=["temporada_atual", "rodada_atual"],
    how="left"
)

In [52]:
df.drop(columns=["posicao_corrida_atual", "posicao_anterior", "posicao_ultima_corrida_x", "status_x", "pontos_anterior_individual_x"], inplace=True)
df.rename(columns={"posicao_ultima_corrida_y": "posicao_ultima_corrida", "status_y": "status", "pontos_anterior_individual_y": "pontos_anterior_individual"}, inplace=True)


In [56]:
import os
caminho_pasta = r'DATA'
arquivos_e_pastas = os.listdir(caminho_pasta)

dfs = []

for item in arquivos_e_pastas:
    if item.startswith("f1_"):
        df = pd.read_csv(f"DATA/{item}")
        dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)

In [57]:
df_final.to_csv(r"DATA\oficial.csv", index=False)